In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.recommender.knn import ItemKNN
from src.recommender.evaluation import precision_at_k, f1_at_k

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Baseline": "#8c8c8c", "KNN": "#4c72b0"}
K_MAX = 10

In [ ]:
train = pd.read_csv(ROOT / "data/processed/train.csv")
test = pd.read_csv(ROOT / "data/processed/test.csv")

actual = test[test["event"] == "transaction"].groupby("visitorid")["itemid"].apply(set).to_dict()
seen = train.groupby("visitorid")["itemid"].apply(set).to_dict()
users = sorted(set(actual) & set(seen))

popular = train[train["event"] == "transaction"].groupby("itemid").size().sort_values(ascending=False).index.tolist()

knn = ItemKNN().fit(train)

recs = {"Baseline": {}, "KNN": {}}
for u in users:
    recs["Baseline"][u] = [int(i) for i in popular if i not in knn.user_purchased.get(u, set())][:K_MAX]
    recs["KNN"][u] = knn.recommend(u, k=K_MAX)

print("Users evaluated:", len(users))

In [ ]:
def per_user_metrics(model, k):
    rows = []
    for u in users:
        r, a = recs[model][u][:k], actual[u]
        hits = len(set(r) & a)
        rows.append({"user": u, "model": model, "k": k, "hits": hits,
                     "precision": precision_at_k(r, a, k), "recall": hits / len(a),
                     "hit": float(hits > 0), "f1": f1_at_k(r, a, k)})
    return pd.DataFrame(rows)

metrics = pd.concat([per_user_metrics(m, k) for m in recs for k in range(1, K_MAX + 1)], ignore_index=True)
metrics.groupby(["model", "k"])[["precision", "recall", "hit", "f1"]].mean().unstack(0).round(4).loc[[1, 3, 5, 10]]

In [ ]:
rng = np.random.default_rng(0)

def bootstrap_ci(values, n=2000):
    values = np.asarray(values)
    means = [rng.choice(values, len(values)).mean() for _ in range(n)]
    return values.mean(), *np.percentile(means, [2.5, 97.5])

k = 5
names = {"precision": "Precision@5", "recall": "Recall@5", "hit": "HitRate@5", "f1": "F1@5"}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, (col, title) in zip(axes, names.items()):
    for i, model in enumerate(["Baseline", "KNN"]):
        m, lo, hi = bootstrap_ci(metrics[(metrics.model == model) & (metrics.k == k)][col])
        ax.bar(i, m, color=COLORS[model], yerr=[[m - lo], [hi - m]], capsize=5)
        ax.text(i, hi, f"{m:.4f}", ha="center", va="bottom")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Baseline", "KNN"]); ax.set_title(title)
fig.suptitle(f"Metrics at k={k} with 95% bootstrap CI (n={len(users)} users)")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (col, title) in zip(axes, [("precision", "Precision@k"), ("recall", "Recall@k"), ("hit", "HitRate@k")]):
    for model in recs:
        s = metrics[metrics.model == model].groupby("k")[col].mean()
        ax.plot(s.index, s.values, marker="o", color=COLORS[model], label=model)
    ax.set_title(title); ax.set_xlabel("k"); ax.legend()
plt.tight_layout()

In [ ]:
k = 5
a = metrics[(metrics.model == "Baseline") & (metrics.k == k)].set_index("user")["hits"]
b = metrics[(metrics.model == "KNN") & (metrics.k == k)].set_index("user")["hits"]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
dist = pd.DataFrame({"Baseline": a.clip(upper=3).value_counts(), "KNN": b.clip(upper=3).value_counts()}).fillna(0).sort_index()
dist.index = [str(i) if i < 3 else "3+" for i in dist.index]
dist.plot.bar(ax=ax[0], color=[COLORS["Baseline"], COLORS["KNN"]], rot=0, title="Users by number of hits in top-5")
ax[0].set_xlabel("hits")

diff = b - a
outcome = pd.Series({"KNN better": (diff > 0).sum(), "tie": (diff == 0).sum(), "Baseline better": (diff < 0).sum()})
outcome.plot.bar(ax=ax[1], color=[COLORS["KNN"], "#cccccc", COLORS["Baseline"]], rot=0, title="Per-user comparison (hits in top-5)")
for i, v in enumerate(outcome):
    ax[1].text(i, v, str(v), ha="center", va="bottom")
ax[1].set_yscale("symlog")
plt.tight_layout()
print("Total hits - Baseline:", int(a.sum()), " KNN:", int(b.sum()))

In [ ]:
hist_len = pd.Series({u: len(seen[u]) for u in users}, name="history")
bins = pd.cut(hist_len, [0, 1, 2, 5, 10, np.inf], labels=["1", "2", "3-5", "6-10", "11+"])
by_hist = metrics[metrics.k == 5].assign(bucket=metrics[metrics.k == 5]["user"].map(bins))
tbl = by_hist.groupby(["bucket", "model"], observed=True)["hit"].mean().unstack()
cnt = by_hist[by_hist.model == "KNN"].groupby("bucket", observed=True).size()

ax = tbl.plot.bar(color=[COLORS["Baseline"], COLORS["KNN"]], rot=0, figsize=(8, 3.8), title="HitRate@5 by number of distinct items seen in train")
for i, n in enumerate(cnt):
    ax.text(i, 0, f"n={n}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("distinct items in user's train history")

In [ ]:
in_cand = set(knn.items.tolist())
rows = []
for u in users:
    a_u = actual[u]
    rows.append({"purchases": len(a_u),
                 "in_knn_candidates": len(a_u & in_cand),
                 "seen_in_train": len(a_u & seen[u]),
                 "in_popular_top5": len(a_u & set(popular[:5]))})
cover = pd.DataFrame(rows).sum()
ax = (cover / cover["purchases"]).drop("purchases").plot.bar(color="#55a868", rot=0, figsize=(8, 3.8),
                                                                title="Share of test purchases that could be hit")
for i, v in enumerate(cover.drop("purchases")):
    ax.text(i, v / cover["purchases"], f"{v}/{cover['purchases']}", ha="center", va="bottom")